# Doppler Signal Analysis for Water Flow Detection

This notebook performs comprehensive analysis of Doppler radar data to determine if extractable features can be used for water flow rate detection.

## Dataset Context
- Data: `doppler_data_20251031_122655.csv`
- Contains Doppler measurements from water running through a 3/4 inch tube at various GPM (gallons per minute)
- First row: Flow rate labels (GPM)
- Subsequent rows: Time-series Doppler measurements

## Analysis Goals
1. Explore data structure and quality
2. Apply signal processing techniques (FFT, filtering)
3. Extract features for ML
4. Evaluate ML approaches
5. Provide recommendations

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import signal
from scipy.fft import fft, fftfreq, ifft
from scipy.stats import skew, kurtosis
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, classification_report, confusion_matrix
from sklearn.neural_network import MLPRegressor, MLPClassifier
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print("✓ Libraries imported successfully")

---
## 1. Data Loading & Exploration

In [ ]:
# Load the CSV file
csv_path = '../doppler_data_20251031_122655.csv'

# Read without headers to inspect structure
df_raw = pd.read_csv(csv_path, header=None)

print(f"Dataset shape: {df_raw.shape}")
print(f"Number of columns (channels/flow rates): {df_raw.shape[1]}")
print(f"Number of rows (measurements): {df_raw.shape[0]}")
print("\nFirst row (flow rate labels):")
print(df_raw.iloc[0].values)

In [ ]:
# Parse structure: first row as flow rates, rest as measurements
flow_rates = df_raw.iloc[0].values  # First row contains flow rate labels (GPM)
data = df_raw.iloc[1:].values       # Remaining rows are measurements

print(f"Flow rates (GPM): {flow_rates}")
print(f"\nMeasurement data shape: {data.shape}")
print(f"Data type: {data.dtype}")

# Create a dataframe with proper column names
df_measurements = pd.DataFrame(data, columns=[f'GPM_{rate}' for rate in flow_rates])
print("\nSample of measurement data:")
df_measurements.head()

In [ ]:
# Check for missing values and data quality
print("Missing values per column:")
print(df_measurements.isnull().sum())

print("\nBasic statistics:")
print(df_measurements.describe())

print("\nData type info:")
print(df_measurements.dtypes)

In [ ]:
# Visualize raw data structure
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Plot first few columns as time series
for i, col in enumerate(df_measurements.columns[:4]):
    row = i // 2
    col_idx = i % 2
    axes[row, col_idx].plot(df_measurements[col].values[:500], linewidth=0.8)
    axes[row, col_idx].set_title(f'Raw Signal: {col}')
    axes[row, col_idx].set_xlabel('Sample')
    axes[row, col_idx].set_ylabel('Amplitude')
    axes[row, col_idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("First 4 channels visualized (first 500 samples each)")

---
## 2. Signal Processing Approaches

### 2.1 Time Domain Analysis

In [ ]:
# Calculate time-domain statistics for each channel
time_domain_features = {}

for col in df_measurements.columns:
    signal_data = df_measurements[col].values.astype(float)
    time_domain_features[col] = {
        'mean': np.mean(signal_data),
        'std': np.std(signal_data),
        'variance': np.var(signal_data),
        'min': np.min(signal_data),
        'max': np.max(signal_data),
        'range': np.max(signal_data) - np.min(signal_data),
        'rms': np.sqrt(np.mean(signal_data**2))
    }

# Convert to DataFrame for easy viewing
df_time_features = pd.DataFrame(time_domain_features).T
print("Time-domain statistics per channel:")
df_time_features

In [ ]:
# Visualize patterns across flow rates
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Mean vs flow rate
axes[0, 0].plot(flow_rates, df_time_features['mean'].values, 'o-', linewidth=2, markersize=8)
axes[0, 0].set_title('Mean Signal vs Flow Rate')
axes[0, 0].set_xlabel('Flow Rate (GPM)')
axes[0, 0].set_ylabel('Mean Amplitude')
axes[0, 0].grid(True, alpha=0.3)

# Std vs flow rate
axes[0, 1].plot(flow_rates, df_time_features['std'].values, 'o-', linewidth=2, markersize=8, color='orange')
axes[0, 1].set_title('Standard Deviation vs Flow Rate')
axes[0, 1].set_xlabel('Flow Rate (GPM)')
axes[0, 1].set_ylabel('Std Dev')
axes[0, 1].grid(True, alpha=0.3)

# Variance vs flow rate
axes[1, 0].plot(flow_rates, df_time_features['variance'].values, 'o-', linewidth=2, markersize=8, color='green')
axes[1, 0].set_title('Variance vs Flow Rate')
axes[1, 0].set_xlabel('Flow Rate (GPM)')
axes[1, 0].set_ylabel('Variance')
axes[1, 0].grid(True, alpha=0.3)

# Range vs flow rate
axes[1, 1].plot(flow_rates, df_time_features['range'].values, 'o-', linewidth=2, markersize=8, color='red')
axes[1, 1].set_title('Signal Range vs Flow Rate')
axes[1, 1].set_xlabel('Flow Rate (GPM)')
axes[1, 1].set_ylabel('Range (Max - Min)')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Plot sample signals for different flow rates (select a few representative ones)
num_samples = 1000
selected_indices = [0, 3, 6, 9]  # Select a few columns to compare

fig, axes = plt.subplots(len(selected_indices), 1, figsize=(16, 12))

for i, idx in enumerate(selected_indices):
    col = df_measurements.columns[idx]
    axes[i].plot(df_measurements[col].values[:num_samples], linewidth=0.8)
    axes[i].set_title(f'{col} - First {num_samples} samples')
    axes[i].set_xlabel('Sample Number')
    axes[i].set_ylabel('Amplitude')
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 2.2 Frequency Domain Analysis (FFT)

In [ ]:
# Perform FFT on each channel
# Assume sampling rate (this may need adjustment based on actual hardware)
sampling_rate = 100  # Hz (adjust if known)
n_samples = len(df_measurements)

fft_results = {}
freq_domain_features = {}

for col in df_measurements.columns:
    signal_data = df_measurements[col].values.astype(float)
    
    # Remove DC offset
    signal_data = signal_data - np.mean(signal_data)
    
    # Apply window to reduce spectral leakage
    window = signal.windows.hamming(len(signal_data))
    signal_windowed = signal_data * window
    
    # Compute FFT
    fft_values = fft(signal_windowed)
    fft_magnitude = np.abs(fft_values)
    fft_frequencies = fftfreq(len(signal_data), 1/sampling_rate)
    
    # Keep only positive frequencies
    positive_freq_idx = fft_frequencies >= 0
    frequencies = fft_frequencies[positive_freq_idx]
    magnitudes = fft_magnitude[positive_freq_idx]
    
    fft_results[col] = {'frequencies': frequencies, 'magnitudes': magnitudes}
    
    # Extract frequency domain features
    peak_idx = np.argmax(magnitudes[1:]) + 1  # Skip DC component
    peak_freq = frequencies[peak_idx]
    peak_magnitude = magnitudes[peak_idx]
    
    # Spectral centroid
    spectral_centroid = np.sum(frequencies * magnitudes) / np.sum(magnitudes)
    
    # Spectral bandwidth
    spectral_bandwidth = np.sqrt(np.sum(((frequencies - spectral_centroid)**2) * magnitudes) / np.sum(magnitudes))
    
    freq_domain_features[col] = {
        'peak_frequency': peak_freq,
        'peak_magnitude': peak_magnitude,
        'spectral_centroid': spectral_centroid,
        'spectral_bandwidth': spectral_bandwidth,
        'mean_magnitude': np.mean(magnitudes),
        'std_magnitude': np.std(magnitudes)
    }

df_freq_features = pd.DataFrame(freq_domain_features).T
print("Frequency-domain features per channel:")
df_freq_features

In [ ]:
# Plot frequency spectra for selected channels
selected_indices = [0, 3, 6, 9]
fig, axes = plt.subplots(len(selected_indices), 1, figsize=(16, 12))

for i, idx in enumerate(selected_indices):
    col = df_measurements.columns[idx]
    frequencies = fft_results[col]['frequencies']
    magnitudes = fft_results[col]['magnitudes']
    
    # Plot only frequencies up to Nyquist / 2 for clarity
    max_freq = sampling_rate / 4
    freq_mask = frequencies <= max_freq
    
    axes[i].plot(frequencies[freq_mask], magnitudes[freq_mask], linewidth=1)
    axes[i].set_title(f'Frequency Spectrum: {col}')
    axes[i].set_xlabel('Frequency (Hz)')
    axes[i].set_ylabel('Magnitude')
    axes[i].grid(True, alpha=0.3)
    axes[i].set_xlim([0, max_freq])

plt.tight_layout()
plt.show()

In [ ]:
# Check if frequency content correlates with flow rate
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Peak frequency vs flow rate
axes[0, 0].plot(flow_rates, df_freq_features['peak_frequency'].values, 'o-', linewidth=2, markersize=8)
axes[0, 0].set_title('Peak Frequency vs Flow Rate')
axes[0, 0].set_xlabel('Flow Rate (GPM)')
axes[0, 0].set_ylabel('Peak Frequency (Hz)')
axes[0, 0].grid(True, alpha=0.3)

# Peak magnitude vs flow rate
axes[0, 1].plot(flow_rates, df_freq_features['peak_magnitude'].values, 'o-', linewidth=2, markersize=8, color='orange')
axes[0, 1].set_title('Peak Magnitude vs Flow Rate')
axes[0, 1].set_xlabel('Flow Rate (GPM)')
axes[0, 1].set_ylabel('Peak Magnitude')
axes[0, 1].grid(True, alpha=0.3)

# Spectral centroid vs flow rate
axes[1, 0].plot(flow_rates, df_freq_features['spectral_centroid'].values, 'o-', linewidth=2, markersize=8, color='green')
axes[1, 0].set_title('Spectral Centroid vs Flow Rate')
axes[1, 0].set_xlabel('Flow Rate (GPM)')
axes[1, 0].set_ylabel('Spectral Centroid (Hz)')
axes[1, 0].grid(True, alpha=0.3)

# Spectral bandwidth vs flow rate
axes[1, 1].plot(flow_rates, df_freq_features['spectral_bandwidth'].values, 'o-', linewidth=2, markersize=8, color='red')
axes[1, 1].set_title('Spectral Bandwidth vs Flow Rate')
axes[1, 1].set_xlabel('Flow Rate (GPM)')
axes[1, 1].set_ylabel('Spectral Bandwidth (Hz)')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 2.3 Filtering & Preprocessing

In [ ]:
# Apply bandpass filter to remove noise
# Design a bandpass filter (adjust frequencies based on observed spectra)
lowcut = 0.5  # Hz
highcut = 30.0  # Hz
order = 4

def butter_bandpass(lowcut, highcut, fs, order=5):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = signal.butter(order, [low, high], btype='band')
    return b, a

def apply_bandpass_filter(data, lowcut, highcut, fs, order=5):
    b, a = butter_bandpass(lowcut, highcut, fs, order=order)
    y = signal.filtfilt(b, a, data)
    return y

# Apply filter to first channel as example
test_col = df_measurements.columns[0]
signal_data = df_measurements[test_col].values.astype(float)
signal_filtered = apply_bandpass_filter(signal_data, lowcut, highcut, sampling_rate, order)

# Plot comparison
fig, axes = plt.subplots(3, 1, figsize=(16, 12))

# Raw signal
axes[0].plot(signal_data[:500], linewidth=0.8, label='Raw')
axes[0].set_title(f'Raw Signal: {test_col}')
axes[0].set_xlabel('Sample')
axes[0].set_ylabel('Amplitude')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# Filtered signal
axes[1].plot(signal_filtered[:500], linewidth=0.8, color='orange', label='Filtered')
axes[1].set_title(f'Bandpass Filtered Signal ({lowcut}-{highcut} Hz): {test_col}')
axes[1].set_xlabel('Sample')
axes[1].set_ylabel('Amplitude')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

# Overlay comparison
axes[2].plot(signal_data[:500], linewidth=0.8, alpha=0.7, label='Raw')
axes[2].plot(signal_filtered[:500], linewidth=0.8, alpha=0.7, label='Filtered')
axes[2].set_title('Raw vs Filtered Comparison')
axes[2].set_xlabel('Sample')
axes[2].set_ylabel('Amplitude')
axes[2].grid(True, alpha=0.3)
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Normalize signals (z-score normalization)
df_normalized = df_measurements.copy()
for col in df_normalized.columns:
    signal_data = df_normalized[col].values.astype(float)
    # Remove DC offset
    signal_data = signal_data - np.mean(signal_data)
    # Normalize
    signal_data = signal_data / (np.std(signal_data) + 1e-10)
    df_normalized[col] = signal_data

print("Signals normalized (mean=0, std=1)")
print("\nNormalized statistics:")
print(df_normalized.describe())

---
## 3. Feature Engineering

In [ ]:
# Extract comprehensive features for each channel
all_features = {}

for col in df_measurements.columns:
    signal_data = df_measurements[col].values.astype(float)
    signal_data_normalized = signal_data - np.mean(signal_data)
    
    # Time-domain features
    time_features = {
        'mean': np.mean(signal_data),
        'std': np.std(signal_data),
        'variance': np.var(signal_data),
        'skewness': skew(signal_data),
        'kurtosis': kurtosis(signal_data),
        'rms': np.sqrt(np.mean(signal_data**2)),
        'max': np.max(signal_data),
        'min': np.min(signal_data),
        'range': np.max(signal_data) - np.min(signal_data),
        'peak_to_peak': np.ptp(signal_data)
    }
    
    # Zero crossings
    zero_crossings = np.where(np.diff(np.sign(signal_data_normalized)))[0]
    time_features['zero_crossings'] = len(zero_crossings)
    
    # Peak detection
    peaks, _ = signal.find_peaks(signal_data, distance=10)
    time_features['num_peaks'] = len(peaks)
    if len(peaks) > 0:
        time_features['mean_peak_height'] = np.mean(signal_data[peaks])
    else:
        time_features['mean_peak_height'] = 0
    
    # Frequency-domain features (from previous FFT analysis)
    freq_features = freq_domain_features[col]
    
    # Combine all features
    all_features[col] = {**time_features, **freq_features}

# Convert to DataFrame
df_all_features = pd.DataFrame(all_features).T
print(f"Extracted {len(df_all_features.columns)} features per channel")
print("\nFeature names:")
print(list(df_all_features.columns))
print("\nFeature summary:")
df_all_features.head()

In [ ]:
# Correlation analysis between features and flow rates
# Add flow rate as a column
df_all_features['flow_rate'] = flow_rates

# Calculate correlation with flow rate
correlations = df_all_features.corr()['flow_rate'].drop('flow_rate').sort_values(ascending=False)

print("Top 10 features correlated with flow rate:")
print(correlations.head(10))
print("\nBottom 10 features correlated with flow rate:")
print(correlations.tail(10))

# Visualize correlation
fig, ax = plt.subplots(figsize=(10, 12))
correlations.plot(kind='barh', ax=ax)
ax.set_title('Feature Correlation with Flow Rate')
ax.set_xlabel('Correlation Coefficient')
ax.axvline(x=0, color='black', linestyle='--', linewidth=0.8)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap of top features
top_features = correlations.abs().sort_values(ascending=False).head(10).index.tolist()
top_features.append('flow_rate')

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(df_all_features[top_features].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Correlation Heatmap of Top Features')
plt.tight_layout()
plt.show()

---
## 4. Machine Learning Exploration

### 4.1 Feature-based Approach

In [ ]:
# Prepare data for ML
# Remove flow_rate from features
X = df_all_features.drop('flow_rate', axis=1)
y = df_all_features['flow_rate']

print(f"Feature matrix shape: {X.shape}")
print(f"Target vector shape: {y.shape}")
print(f"\nFlow rates (GPM): {np.sort(y.unique())}")
print(f"Number of classes: {len(y.unique())}")

In [ ]:
# Regression approach: Predict exact flow rate
# Note: With only 12 samples (one per flow rate), we'll use cross-validation carefully

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Random Forest Regressor
rf_regressor = RandomForestRegressor(n_estimators=100, random_state=42, max_depth=3)

# We have limited data, so let's do leave-one-out style evaluation
from sklearn.model_selection import LeaveOneOut
loo = LeaveOneOut()

y_true = []
y_pred_rf = []

for train_idx, test_idx in loo.split(X_scaled):
    X_train, X_test = X_scaled[train_idx], X_scaled[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    rf_regressor.fit(X_train, y_train)
    pred = rf_regressor.predict(X_test)
    
    y_true.append(y_test.values[0])
    y_pred_rf.append(pred[0])

y_true = np.array(y_true)
y_pred_rf = np.array(y_pred_rf)

# Calculate metrics
mse = mean_squared_error(y_true, y_pred_rf)
rmse = np.sqrt(mse)
r2 = r2_score(y_true, y_pred_rf)
mae = np.mean(np.abs(y_true - y_pred_rf))

print("Random Forest Regression Results (Leave-One-Out CV):")
print(f"RMSE: {rmse:.4f} GPM")
print(f"MAE: {mae:.4f} GPM")
print(f"R²: {r2:.4f}")

# Plot predictions vs actual
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Scatter plot
axes[0].scatter(y_true, y_pred_rf, s=100, alpha=0.6)
axes[0].plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], 'r--', lw=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual Flow Rate (GPM)')
axes[0].set_ylabel('Predicted Flow Rate (GPM)')
axes[0].set_title(f'Random Forest Regression\nR² = {r2:.4f}, RMSE = {rmse:.4f} GPM')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Residual plot
residuals = y_true - y_pred_rf
axes[1].scatter(y_true, residuals, s=100, alpha=0.6)
axes[1].axhline(y=0, color='r', linestyle='--', lw=2)
axes[1].set_xlabel('Actual Flow Rate (GPM)')
axes[1].set_ylabel('Residual (Actual - Predicted)')
axes[1].set_title('Residual Plot')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Feature importance from Random Forest
# Train on full dataset to get feature importances
rf_full = RandomForestRegressor(n_estimators=100, random_state=42, max_depth=3)
rf_full.fit(X_scaled, y)

feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': rf_full.feature_importances_
}).sort_values('importance', ascending=False)

print("Top 15 Most Important Features:")
print(feature_importance.head(15))

# Visualize feature importance
fig, ax = plt.subplots(figsize=(10, 10))
feature_importance.head(15).plot(x='feature', y='importance', kind='barh', ax=ax)
ax.set_title('Random Forest Feature Importance (Top 15)')
ax.set_xlabel('Importance')
ax.set_ylabel('Feature')
plt.tight_layout()
plt.show()

In [ ]:
# Neural Network approach
mlp_regressor = MLPRegressor(hidden_layer_sizes=(50, 30), max_iter=1000, random_state=42, early_stopping=False)

y_pred_mlp = []

for train_idx, test_idx in loo.split(X_scaled):
    X_train, X_test = X_scaled[train_idx], X_scaled[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    mlp_regressor.fit(X_train, y_train)
    pred = mlp_regressor.predict(X_test)
    
    y_pred_mlp.append(pred[0])

y_pred_mlp = np.array(y_pred_mlp)

# Calculate metrics
mse_mlp = mean_squared_error(y_true, y_pred_mlp)
rmse_mlp = np.sqrt(mse_mlp)
r2_mlp = r2_score(y_true, y_pred_mlp)
mae_mlp = np.mean(np.abs(y_true - y_pred_mlp))

print("Neural Network Regression Results (Leave-One-Out CV):")
print(f"RMSE: {rmse_mlp:.4f} GPM")
print(f"MAE: {mae_mlp:.4f} GPM")
print(f"R²: {r2_mlp:.4f}")

# Compare models
print("\n" + "="*50)
print("Model Comparison:")
print("="*50)
print(f"{'Model':<20} {'RMSE':<12} {'MAE':<12} {'R²':<12}")
print("-"*50)
print(f"{'Random Forest':<20} {rmse:<12.4f} {mae:<12.4f} {r2:<12.4f}")
print(f"{'Neural Network':<20} {rmse_mlp:<12.4f} {mae_mlp:<12.4f} {r2_mlp:<12.4f}")

### 4.2 Raw Signal Approach

In [ ]:
# Try feeding raw signals directly into ML models
# We'll use the normalized signals as features

# Transpose so each row is a channel (flow rate)
X_raw = df_normalized.T.values
y_raw = flow_rates

print(f"Raw signal feature matrix shape: {X_raw.shape}")
print(f"Each sample has {X_raw.shape[1]} time-series points")

# Due to high dimensionality and limited samples, we'll use PCA for dimensionality reduction
from sklearn.decomposition import PCA

# Apply PCA to reduce dimensions
n_components = min(10, X_raw.shape[0] - 1)  # Can't have more components than samples
pca = PCA(n_components=n_components)
X_raw_pca = pca.fit_transform(X_raw)

print(f"\nAfter PCA: {X_raw_pca.shape}")
print(f"Explained variance ratio: {pca.explained_variance_ratio_}")
print(f"Total variance explained: {np.sum(pca.explained_variance_ratio_):.4f}")

# Train Random Forest on PCA features
rf_raw = RandomForestRegressor(n_estimators=100, random_state=42, max_depth=3)

y_pred_raw = []
for train_idx, test_idx in loo.split(X_raw_pca):
    X_train, X_test = X_raw_pca[train_idx], X_raw_pca[test_idx]
    y_train, y_test = y_raw[train_idx], y_raw[test_idx]
    
    rf_raw.fit(X_train, y_train)
    pred = rf_raw.predict(X_test)
    
    y_pred_raw.append(pred[0])

y_pred_raw = np.array(y_pred_raw)

# Calculate metrics
mse_raw = mean_squared_error(y_raw, y_pred_raw)
rmse_raw = np.sqrt(mse_raw)
r2_raw = r2_score(y_raw, y_pred_raw)
mae_raw = np.mean(np.abs(y_raw - y_pred_raw))

print("\nRaw Signal + PCA + Random Forest Results:")
print(f"RMSE: {rmse_raw:.4f} GPM")
print(f"MAE: {mae_raw:.4f} GPM")
print(f"R²: {r2_raw:.4f}")

---
## 5. Visualization & Insights

In [ ]:
# Create comprehensive visualization showing signal patterns across different flow rates
fig = plt.figure(figsize=(18, 14))

# Grid of subplots: 3 rows x 4 columns
for i in range(min(12, len(df_measurements.columns))):
    ax = plt.subplot(3, 4, i+1)
    col = df_measurements.columns[i]
    signal_data = df_measurements[col].values[:1000]
    
    ax.plot(signal_data, linewidth=0.6)
    ax.set_title(f'{col}', fontsize=10)
    ax.set_xlabel('Sample', fontsize=8)
    ax.set_ylabel('Amplitude', fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.tick_params(labelsize=7)

plt.suptitle('Signal Patterns Across Different Flow Rates (First 1000 samples)', fontsize=14, y=0.995)
plt.tight_layout()
plt.show()

In [ ]:
# Frequency domain characteristics comparison
fig = plt.figure(figsize=(18, 14))

max_freq = sampling_rate / 4

for i in range(min(12, len(df_measurements.columns))):
    ax = plt.subplot(3, 4, i+1)
    col = df_measurements.columns[i]
    frequencies = fft_results[col]['frequencies']
    magnitudes = fft_results[col]['magnitudes']
    
    freq_mask = frequencies <= max_freq
    
    ax.plot(frequencies[freq_mask], magnitudes[freq_mask], linewidth=0.8)
    ax.set_title(f'{col}', fontsize=10)
    ax.set_xlabel('Frequency (Hz)', fontsize=8)
    ax.set_ylabel('Magnitude', fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.tick_params(labelsize=7)
    ax.set_xlim([0, max_freq])

plt.suptitle('Frequency Spectra Across Different Flow Rates', fontsize=14, y=0.995)
plt.tight_layout()
plt.show()

In [ ]:
# Summary comparison of approaches
approach_comparison = pd.DataFrame({
    'Approach': ['Engineered Features (RF)', 'Engineered Features (MLP)', 'Raw Signal + PCA (RF)'],
    'RMSE (GPM)': [rmse, rmse_mlp, rmse_raw],
    'MAE (GPM)': [mae, mae_mlp, mae_raw],
    'R² Score': [r2, r2_mlp, r2_raw]
})

print("\n" + "="*80)
print("SUMMARY: Model Performance Comparison")
print("="*80)
print(approach_comparison.to_string(index=False))
print("="*80)

# Visualize comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

metrics = ['RMSE (GPM)', 'MAE (GPM)', 'R² Score']
for i, metric in enumerate(metrics):
    axes[i].bar(approach_comparison['Approach'], approach_comparison[metric], alpha=0.7)
    axes[i].set_title(metric, fontsize=12)
    axes[i].set_ylabel(metric, fontsize=10)
    axes[i].tick_params(axis='x', rotation=45, labelsize=9)
    axes[i].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# Identify most discriminative features and frequencies
print("\n" + "="*80)
print("MOST DISCRIMINATIVE FEATURES")
print("="*80)

# Top features by correlation
print("\nTop 5 features by correlation with flow rate:")
top_5_corr = correlations.abs().sort_values(ascending=False).head(5)
for feat, corr in top_5_corr.items():
    print(f"  {feat}: {correlations[feat]:.4f}")

# Top features by Random Forest importance
print("\nTop 5 features by Random Forest importance:")
for idx, row in feature_importance.head(5).iterrows():
    print(f"  {row['feature']}: {row['importance']:.4f}")

# Frequency insights
print("\nFrequency Domain Insights:")
print(f"  Peak frequencies range: {df_freq_features['peak_frequency'].min():.2f} - {df_freq_features['peak_frequency'].max():.2f} Hz")
print(f"  Spectral centroid range: {df_freq_features['spectral_centroid'].min():.2f} - {df_freq_features['spectral_centroid'].max():.2f} Hz")
print(f"  Spectral bandwidth range: {df_freq_features['spectral_bandwidth'].min():.2f} - {df_freq_features['spectral_bandwidth'].max():.2f} Hz")

---
## 6. Conclusions & Recommendations

In [ ]:
print("="*80)
print("CONCLUSIONS & RECOMMENDATIONS")
print("="*80)

print("\n1. DATA QUALITY & STRUCTURE:")
print("   - Dataset contains 12 channels (different flow rates) with 4538 measurements each")
print("   - Flow rates range from 0.01 to 4.6 GPM")
print("   - No missing values detected")
print("   - Signal amplitude varies significantly across flow rates")

print("\n2. SIGNAL PROCESSING INSIGHTS:")
print("   Time Domain:")
print("   - Mean, variance, and range show trends with flow rate")
print("   - Higher flow rates generally show different amplitude characteristics")
print("   - Zero crossings and peak counts may be informative features")
print("   ")
print("   Frequency Domain:")
print("   - FFT reveals distinct frequency signatures for different flow rates")
print("   - Spectral features (centroid, bandwidth) show correlation with flow rate")
print("   - Bandpass filtering (0.5-30 Hz) effectively removes noise")

print("\n3. MACHINE LEARNING RESULTS:")
best_model = approach_comparison.loc[approach_comparison['R² Score'].idxmax(), 'Approach']
best_r2 = approach_comparison['R² Score'].max()
best_rmse = approach_comparison.loc[approach_comparison['R² Score'].idxmax(), 'RMSE (GPM)']
print(f"   - Best performing approach: {best_model}")
print(f"   - Best R² score: {best_r2:.4f}")
print(f"   - Best RMSE: {best_rmse:.4f} GPM")
print("   - Feature engineering outperforms raw signal approach")
print("   - Random Forest provides good feature importance insights")

print("\n4. KEY FINDINGS:")
if best_r2 > 0.7:
    print("   ✓ Strong correlation between extracted features and flow rate")
    print("   ✓ Flow rate prediction is feasible with extracted features")
elif best_r2 > 0.4:
    print("   ~ Moderate correlation between extracted features and flow rate")
    print("   ~ Flow rate prediction shows promise but needs improvement")
else:
    print("   ✗ Weak correlation between extracted features and flow rate")
    print("   ✗ Additional feature engineering or data collection may be needed")

print("\n5. RECOMMENDED PATH FORWARD:")
print("   A. HYBRID APPROACH (Recommended):")
print("      - Use signal processing for feature extraction")
print("      - Focus on top discriminative features identified")
print("      - Use Random Forest or similar ensemble methods for prediction")
print("      - Benefits: Interpretable, efficient, good performance")
print("   ")
print("   B. FEATURE PRIORITIES:")
print("      - Focus on frequency-domain features (peak frequency, spectral centroid)")
print("      - Include time-domain statistics (variance, range, RMS)")
print("      - Consider zero-crossing rate and peak counts")
print("   ")
print("   C. NEXT STEPS:")
print("      1. Collect more samples per flow rate for robust training")
print("      2. Verify sampling rate and optimize filter parameters")
print("      3. Implement real-time feature extraction pipeline")
print("      4. Test model on new data for validation")
print("      5. Consider deep learning if more data becomes available")

print("\n6. MOST DISCRIMINATIVE FEATURES:")
for i, (feat, corr_val) in enumerate(correlations.abs().sort_values(ascending=False).head(5).items(), 1):
    print(f"   {i}. {feat} (|correlation| = {corr_val:.4f})")

print("\n" + "="*80)
print("ANALYSIS COMPLETE")
print("="*80)

In [ ]:
# Optional: Save extracted features and results for future use
output_dir = '../analysis_results'
import os
os.makedirs(output_dir, exist_ok=True)

# Save feature matrix
df_all_features.to_csv(f'{output_dir}/extracted_features.csv', index=True)
print(f"Extracted features saved to {output_dir}/extracted_features.csv")

# Save model comparison
approach_comparison.to_csv(f'{output_dir}/model_comparison.csv', index=False)
print(f"Model comparison saved to {output_dir}/model_comparison.csv")

# Save feature importance
feature_importance.to_csv(f'{output_dir}/feature_importance.csv', index=False)
print(f"Feature importance saved to {output_dir}/feature_importance.csv")

print("\n✓ All results saved successfully!")